In [2]:
import os
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser


In [3]:
load_dotenv()


True

In [4]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    #model="gemini-1.5-flash",
    temperature=0
)



In [5]:
job_description = """
Looking for a Data Scientist with:
- Python
- Machine Learning
- Pandas, NumPy
- SQL
- Data Visualization
- 2+ years experience
"""


In [6]:
resume_strong = """
3 years experience as Data Scientist.
Skills: Python, Machine Learning, Pandas, NumPy, SQL, Tableau
Built ML models for prediction.
"""

resume_avg = """
1 year experience in data analysis.
Skills: Python, Pandas, basic SQL
Worked on small projects.
"""

resume_weak = """
Fresher.
Skills: MS Excel, Communication
No experience in data science.
"""


In [7]:
extract_prompt = PromptTemplate.from_template("""
Extract the following from the resume:
- Skills
- Experience
- Tools

Resume:
{resume}

Rules:
- Do NOT assume anything
- Only extract from given text
""")

extract_chain = extract_prompt | llm | StrOutputParser()


In [8]:
final_prompt = PromptTemplate.from_template("""
You are an AI Resume Screening System.

Job Description:
{jd}

Resume:
{resume}

Tasks:
1. Extract skills, experience, tools
2. Compare with job description
3. Give score (0-100)
4. Explain the reason

Rules:
- Do NOT assume missing skills
- Be accurate

Return in format:
Skills:
Match:
Score:
Explanation:
""")


In [9]:
final_chain = final_prompt | llm | StrOutputParser()

def run_pipeline(resume):
    return final_chain.invoke({
        "jd": job_description,
        "resume": resume
    })


In [10]:
print("STRONG CANDIDATE\n", run_pipeline(resume_strong))



STRONG CANDIDATE
 Skills: Python, Machine Learning, Pandas, NumPy, SQL, Tableau
Match: Excellent
Score: 100
Explanation:
The candidate's resume perfectly matches all the requirements outlined in the job description.
*   **Experience:** The candidate has 3 years of experience, exceeding the "2+ years experience" requirement.
*   **Python:** Explicitly listed.
*   **Machine Learning:** Explicitly listed, and further supported by "Built ML models for prediction."
*   **Pandas, NumPy:** Explicitly listed.
*   **SQL:** Explicitly listed.
*   **Data Visualization:** While "Data Visualization" isn't explicitly listed as a skill, "Tableau" is listed as a tool. Tableau is a leading data visualization tool, indicating proficiency in data visualization.


In [ ]:
print("\nAVERAGE CANDIDATE\n", run_pipeline(resume_avg))


In [ ]:
print("\nWEAK CANDIDATE\n", run_pipeline(resume_weak))